# EEG Model Optimization Pipeline

This notebook optimizes all EEG classification models using GPU acceleration.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Run optimization to find best parameters
4. Evaluate all models with optimal parameters
5. Download results

## Setup

In [ ]:
# Mount Google Drive for results saving
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q mne pyeeg

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Navigate to repository and set up data path
import os
os.chdir('/content/eye-eeg')  # Repository folder
print(f"Repository directory: {os.getcwd()}")

# Data should be in /content/data (next to repository)
data_path = '/content/data'
if os.path.exists(data_path):
    print(f"Data directory found: {data_path}")
else:
    print(f"⚠️  Data directory not found at {data_path}")
    print("Please upload your data folder to /content/data")

In [ ]:
# Import optimization functions
import sys
sys.path.append('/content/eye-eeg')

# Test imports
try:
    from classification.model_optimization import (
        optimize_all_models, evaluate_all_models, 
        get_data_path, get_results_path, clear_data_cache,
        PARAM_GRIDS
    )
    print("✅ Optimization functions imported successfully")
    
    # Test path configuration
    print(f"Data path: {get_data_path()}")
    print(f"Results path: {get_results_path()}")
    
    # Show available models (currently only feature-based work)
    available_models = [model for model in PARAM_GRIDS.keys() if model.startswith('features_')]
    print(f"Available models: {available_models}")
    
except Exception as e:
    print(f"❌ Import error: {e}")
    import traceback
    traceback.print_exc()

## Step 1: Model Optimization

Find optimal hyperparameters for all models. This may take 30-60 minutes depending on the number of subjects and trials.

In [ ]:
# Quick optimization (3 subjects, 8 trials per model, normal split)
print("=== QUICK OPTIMIZATION (for testing) ===")
quick_subjects = SUBJECTS[:3]

# Clear any existing cache to ensure fresh start
from model_optimization import clear_data_cache
clear_data_cache()

quick_results = optimize_all_models(
    subjects=quick_subjects,
    max_trials=8,
    results_dir="/content/drive/MyDrive/eye-eeg/optimization_results",
    split_strategies=['normal']  # Just normal split for quick test
)

In [ ]:
# Full optimization (all subjects, both split strategies)
# Uncomment to run full optimization

print("=== FULL OPTIMIZATION ===")
print("This will take much longer but give better results")

# Clear cache first
clear_data_cache()

full_results = optimize_all_models(
    subjects=SUBJECTS,
    max_trials=20,
    results_dir="/content/drive/MyDrive/eye-eeg/optimization_results",
    split_strategies=['normal', 'leave_one_out']  # Both strategies
)

## Step 2: Model Evaluation

Train and evaluate all models using the optimized parameters.

In [ ]:
# Evaluate models with optimal parameters
print("=== MODEL EVALUATION ===")

# Choose which split strategy to evaluate
split_strategy = 'normal'  # or 'leave_one_out'

evaluation_results = evaluate_all_models(
    subjects=SUBJECTS,  # Use all subjects for evaluation
    split_strategy=split_strategy,
    results_dir="/content/drive/MyDrive/eye-eeg/optimization_results"
)

## Step 3: Results Analysis

In [ ]:
# Load and display results
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load latest results
results_dir = "/content/drive/MyDrive/eye-eeg/optimization_results"
split_strategy = 'normal'  # Change to 'leave_one_out' if you used that

# Load optimal parameters
param_file = f"{results_dir}/optimal_parameters_{split_strategy}_latest.json"
with open(param_file, 'r') as f:
    optimal_params = json.load(f)

# Load evaluation results  
eval_file = f"{results_dir}/evaluation_results_{split_strategy}_latest.json"
with open(eval_file, 'r') as f:
    eval_results = json.load(f)

# Create results dataframe
results_data = []
for model_name, metrics in eval_results.items():
    results_data.append({
        'Model': model_name,
        'Accuracy': metrics['accuracy'],
        'Balanced Accuracy': metrics['balanced_accuracy'],
        'F1-Macro': metrics['f1_macro']
    })

df_results = pd.DataFrame(results_data)
df_results = df_results.sort_values('Balanced Accuracy', ascending=False)

print(f"=== FINAL RESULTS ({split_strategy.upper()} SPLIT) ===")
print(df_results.round(4))

print(f"\nBest Model: {df_results.iloc[0]['Model']} (Balanced Accuracy: {df_results.iloc[0]['Balanced Accuracy']:.4f})")

In [ ]:
# Plot results
plt.figure(figsize=(15, 5))

# Plot 1: Balanced Accuracy
plt.subplot(1, 3, 1)
plt.bar(df_results['Model'], df_results['Balanced Accuracy'], alpha=0.7)
plt.title('Model Balanced Accuracy')
plt.ylabel('Balanced Accuracy')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 2: F1-Macro
plt.subplot(1, 3, 2)
plt.bar(df_results['Model'], df_results['F1-Macro'], alpha=0.7, color='orange')
plt.title('Model F1-Macro Score')
plt.ylabel('F1-Macro')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 3: Accuracy
plt.subplot(1, 3, 3)
plt.bar(df_results['Model'], df_results['Accuracy'], alpha=0.7, color='green')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Display best parameters for each model
print(f"=== OPTIMAL PARAMETERS ({split_strategy.upper()} SPLIT) ===")
for model_name, param_info in optimal_params.items():
    print(f"\n{model_name}:")
    print(f"  Validation Score: {param_info['score']:.4f}")
    print(f"  Parameters: {param_info['params']}")

# Display RAM usage
import psutil
ram_usage = psutil.virtual_memory()
print(f"\nRAM Usage: {ram_usage.percent:.1f}% ({ram_usage.used/1024**3:.1f}GB / {ram_usage.total/1024**3:.1f}GB)")

## Step 4: Download Results

All results are automatically saved to your Google Drive in the `optimization_results` folder.

Files created:
- `optimal_parameters_latest.json` - Best parameters for each model
- `evaluation_results_latest.json` - Final test performance
- `model_comparison.png` - Results visualization
- Timestamped versions of all files

In [ ]:
# Test channel filtering functionality
def test_channel_filtering():
    """Test the new channel filtering capability"""
    try:
        from data.data_loader import DataLoader
        from data.data_configs import SUBJECTS
        
        # Test with specific channels
        data_loader = DataLoader(data_path='/content/data')
        test_subjects = SUBJECTS[:1]  # Use first subject for testing
        test_channels = ['P7', 'F3', 'Pz']  # Test with specific channels
        
        print(f"Testing channel filtering with subjects: {test_subjects}")
        print(f"Testing channels: {test_channels}")
        
        # Test loading features with channel filtering
        X, y, feature_names, _, _ = data_loader.load_features(
            subjects=test_subjects, 
            mode="advanced",
            channels=test_channels
        )
        
        print(f"✅ Successfully loaded {X.shape[0]} samples with {X.shape[1]} features")
        print(f"Channel-filtered features (first 10): {feature_names[:10]}")
        print(f"Labels shape: {y.shape}")
        
        # Test without channel filtering
        X_all, y_all, feature_names_all, _, _ = data_loader.load_features(
            subjects=test_subjects, 
            mode="advanced",
            channels=None
        )
        
        print(f"✅ All channels: {X_all.shape[1]} features vs filtered: {X.shape[1]} features")
        print(f"Reduction: {(X_all.shape[1] - X.shape[1]) / X_all.shape[1] * 100:.1f}%")
        
        return True
        
    except Exception as e:
        print(f"❌ Channel filtering test failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run the test
if test_channel_filtering():
    print("🎉 Ready for channel-specific optimization!")
else:
    print("⚠️  Fix channel filtering issues before proceeding")